# Day 10 — DAG Metrics: Decision-Based Evaluation

**Module 2 · The Metric Toolkit**

Some evaluation criteria are not simply:

> "How good is this answer?"

Instead, they are a sequence of decisions:

```text
Extract information
       ↓
Check a condition
       ↓
Choose a branch
       ↓
Assign a score

## 1. Setup

We will use the same OpenAI judge and concurrency limit used throughout the course.

In [1]:
import os

from dotenv import load_dotenv
from deepeval.models import OpenAIModel
from deepeval.evaluate import AsyncConfig

load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found."

judge = OpenAIModel(
    model="gpt-4.1-mini",
    temperature=0,
)

async_config = AsyncConfig(
    max_concurrent=2
)

print("Judge:", judge.get_model_name())
print("Max concurrent:", async_config.max_concurrent)

Judge: gpt-4.1-mini
Max concurrent: 2


## 2. A Simple Evaluation Problem

Suppose our application generates a meeting summary.

We require three sections:

- Intro
- Body
- Conclusion

Our evaluation logic is:

```text
Are all three sections present?
          │
       ┌──┴──┐
      NO     YES
      ↓       ↓
   Score 0   Check order
                │
          ┌─────┴─────┐
         Good       Wrong
          ↓            ↓
        1.0          0.5

Notice that the order should only be checked if all required sections exist.

This is where a DAG becomes useful


## 3. DAG Building Blocks

We will use three concepts:

| Node | Purpose |
|---|---|
| `TaskNode` | Performs an extraction/task |
| `BinaryJudgementNode` | Makes a yes/no decision |
| `DAGMetric` | Runs the evaluation graph |

We do not need to learn every DAG node type today.
The important idea is **branching evaluation logic**.

In [3]:
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import (
    TaskNode,
    BinaryJudgementNode,
    DeepAcyclicGraph,
)
from deepeval.test_case import SingleTurnParams

# Step 1: Extract the section headings
extract_headings = TaskNode(
    instructions="Extract the section headings from the actual output.",
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    output_label="Section headings",
)

# Step 2: Check whether all required sections exist
check_sections = BinaryJudgementNode(
    criteria=(
        "Do the headings contain all three required sections: "
        "Intro, Body, and Conclusion?"
    )
)

# Missing sections → score 0
check_sections.add_verdict(
    False,
    score=0,
)

# All sections present → score 1
check_sections.add_verdict(
    True,
    score=1,
)

extract_headings.add_node(check_sections)

dag = DeepAcyclicGraph(
    root_nodes=[extract_headings]
)

metric = DAGMetric(
    name="Required Sections",
    dag=dag,
    model=judge,
    async_mode=False,
    threshold=0.5,
)

print("DAG metric created.")

DAG metric created.


## 4. Create Test Cases

We will test two summaries:

1. One containing all required sections.
2. One missing the Conclusion section.

In [4]:
from deepeval.test_case import LLMTestCase

meeting = (
    'Alice: "Today\'s agenda: product update, blockers, marketing timeline."\n'
    'Bob: "Core features done; optimizing performance. Fixes by Friday."\n'
    'Alice: "Plan: fixes by Friday, sync next Wednesday."'
)

test_cases = [
    LLMTestCase(
        input=meeting,
        actual_output=(
            "Intro:\nAlice outlined the agenda.\n\n"
            "Body:\nBob reported performance optimizations.\n\n"
            "Conclusion:\nThe team aligned on fixes by Friday."
        ),
    ),
    LLMTestCase(
        input=meeting,
        actual_output=(
            "Intro:\nAlice outlined the agenda.\n\n"
            "Body:\nBob reported performance optimizations."
        ),
    ),
]

## 5. Run the DAG Evaluation

Unlike a normal metric, a DAG metric follows a specific path through the evaluation graph.

We can inspect the decision trace using `verbose_logs`.

In [5]:
for i, test_case in enumerate(test_cases, start=1):

    metric.measure(test_case)

    print(f"\n{'=' * 50}")
    print(f"Case {i}")
    print(f"Score:   {metric.score:.2f}")
    print(f"Success: {metric.is_successful()}")

    print("\nDecision trace:")
    print(metric.verbose_logs)

c:\Users\T14s\AppData\Local\Programs\Python\Python311\Lib\site-packages\rich\live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')


Case 1
Score:   0.10
Success: False

Decision trace:
______________________
| TaskNode | Level == 0 |
*******************************
Label: None

Instructions:
Extract the section headings from the actual output.

Section headings:
['Intro', 'Body', 'Conclusion']
 
 
__________________________________
| BinaryJudgementNode | Level == 1 |
************************************************
Label: None

Criteria:
Do the headings contain all three required sections: Intro, Body, and Conclusion?

Verdict: True
Reason: The provided section headings explicitly include 'Intro', 'Body', and 'Conclusion', which are the three required sections.
 
 
________________________
| VerdictNode | Level == 2 |
**********************************
Verdict: True
Type: Deterministic



Case 2
Score:   0.00
Success: False

Decision trace:
______________________
| TaskNode | Level == 0 |
*******************************
Label: None

Instructions:
Extract the section headings from the actual output.

Section headings:
['Intro', 'Body']
 
 
__________________________________
| BinaryJudgementNode | Level == 1 |
************************************************
Label: None

Criteria:
Do the headings contain all three required sections: Intro, Body, and Conclusion?

Verdict: False
Reason: The headings provided are 'Intro' and 'Body', but the 'Conclusion' section is missing.
 
 
________________________
| VerdictNode | Level == 2 |
**********************************
Verdict: False
Type: Deterministic


## 6. DAG vs G-Eval

Both use an LLM judge, but they structure evaluation differently.

### G-Eval

```text
Question
   ↓
Evaluation criteria
   ↓
Judge
   ↓
Score

# Day 10 — Key Takeaways

- A DAG represents evaluation as a sequence of tasks and decisions.
- `TaskNode` performs an evaluation task.
- `BinaryJudgementNode` creates a yes/no branch.
- `DAGMetric` turns the graph into a DeepEval metric.
- DAGs are useful when one evaluation decision depends on another.

The important distinction:

```text
G-Eval → "Describe how good this is."

DAG     → "Follow these evaluation decisions."